# Third Deliverable - NYC Taxi Analysis

In this notebook I am exploring the NYC taxi trip dataset like a mini analyst task. First I understand the dataset, then I work on a smaller sample, compare Pandas vs NumPy, create time-based features, run `groupby()` analysis, and finish with mini EDA insights.


## Setup

I am using the full CSV only to count total rows. For the actual analysis, I will work on a **100,000 row sample** so the notebook stays fast and readable.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from IPython.display import display
pd.set_option("display.max_column", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

csv_path = Path("NYC_taxi.csv")

with csv_path.open("r", encoding="utf-8") as file:
    total_rows = sum(1 for _ in file) - 1

sample = min(100000, total_rows)

sample_df = pd.read_csv(
    csv_path,
    nrows=sample,
    parse_dates=['pickup_datetime','dropoff_datetime'])

pd.DataFrame({
    "metrics": ['Total rows in dataset', 'Rows used for analysis'],
    "value": [total_rows, len(sample_df)]
})



,metrics,value
0,Total rows in dataset,1458644
1,Rows used for analysis,100000


In [2]:
sample_df.isna().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [3]:
sample_df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.98,40.77,-73.96,40.77,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.98,40.74,-74.00,40.73,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.98,40.76,-74.01,40.71,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.01,40.72,-74.01,40.71,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.97,40.79,-73.97,40.78,N,435


##  Pandas vs NumPy

Calculated the same three values using both libraries:

1. Average trip duration
2. Average passenger count
3. Total trips



In [4]:
%%time
pandas_metrics = pd.DataFrame({
    "Method": "pandas",
    "avg_trip_duration_Sec": [sample_df['trip_duration'].mean()],
    "avg_trip_duration_min": [sample_df['trip_duration'].mean() / 60],
    "avg_passenger_count" : [sample_df['passenger_count'].mean()],
    "total_trips": [sample_df.shape[0]]
})

pandas_metrics.round(2)

CPU times: total: 0 ns
Wall time: 1.33 ms


,Method,avg_trip_duration_Sec,avg_trip_duration_min,avg_passenger_count,total_trips
0,pandas,939.86,15.66,1.67,100000


In [5]:
%%time
trip_duration_np = sample_df['trip_duration'].to_numpy()
passenger_count = sample_df['passenger_count'].to_numpy()

numpy_metrics = pd.DataFrame({
    "Method": ['numpy'],
    "avg_trip_duration_sec": [trip_duration_np.mean()],
    "avg_trip_duration_min" : [trip_duration_np.mean()/60],
    "avg_passenger_count": [passenger_count.mean()],
    "total trips" : [trip_duration_np.size]
})

numpy_metrics.round(2)

CPU times: total: 0 ns
Wall time: 1.31 ms


,Method,avg_trip_duration_sec,avg_trip_duration_min,avg_passenger_count,total trips
0,numpy,939.86,15.66,1.67,100000


**Pandas vs NumPy note:** Both gave the same answer. Pandas felt more readable and numpy is faster


## Feature engineering

To do time-based analysis, I created `hour` and `day_name` from `pickup_datetime`. I also created `trip_duration_min` to make duration easier to read in minutes, and simple rounded location grids for a light geography check.



In [6]:
analysis_df = sample_df.copy()
analysis_df['Hour'] = analysis_df['pickup_datetime'].dt.hour
analysis_df['day_name'] = analysis_df['pickup_datetime'].dt.day_name()
analysis_df['trip_duration_min'] = analysis_df['trip_duration'] / 60
analysis_df['pickup_grid'] = analysis_df['pickup_latitude'].round(2).astype(str) + ', ' + analysis_df['pickup_longitude'].round(2).astype(str)
analysis_df['dropoff_grid'] = analysis_df['dropoff_latitude'].round(2).astype(str) + ', ' + analysis_df['dropoff_longitude'].round(2).astype(str)
analysis_df[['pickup_datetime', 'Hour', 'day_name', 'trip_duration' ,'trip_duration_min', 'pickup_grid','dropoff_grid']].head()

,pickup_datetime,Hour,day_name,trip_duration,trip_duration_min,pickup_grid,dropoff_grid
0,2016-03-14 17:24:55,17,Monday,455,7.58,"40.77, -73.98","40.77, -73.96"
1,2016-06-12 00:43:35,0,Sunday,663,11.05,"40.74, -73.98","40.73, -74.0"
2,2016-01-19 11:35:24,11,Tuesday,2124,35.40,"40.76, -73.98","40.71, -74.01"
3,2016-04-06 19:32:31,19,Wednesday,429,7.15,"40.72, -74.01","40.71, -74.01"
4,2016-03-26 13:30:55,13,Saturday,435,7.25,"40.79, -73.97","40.78, -73.97"


## Step 5 - GroupBy (Core)


### Aggregation 1 - Which hour has the highest number of trips?


In [7]:
trip_by_hour = (
    analysis_df.groupby('Hour')
    .size()
    .reset_index(name='total_trips')
    .sort_values('total_trips',ascending=False)
)

trip_by_hour.head(10)

,Hour,total_trips
18,18,6288
19,19,6272
21,21,5849
20,20,5749
22,22,5448
17,17,5307
14,14,5161
12,12,5095
15,15,4959
13,13,4838


**Insight:** Evening hours dominate. In this sample, 6 PM is the peak hour with a little over 6.2k trips, which suggests strong after-work travel demand.


### Aggregation 2 - How does passenger count affect trip duration?


In [8]:
duration_by_passenger = (
    analysis_df.groupby('passenger_count')
    .agg(
        total_trips=('id', 'size'),
        mean_trip_duration_min=('trip_duration_min', 'mean')
    )
    .reset_index()
)
duration_by_passenger.round(2)

,passenger_count,total_trips,mean_trip_duration_min
0,0,1,"1,431.68"
1,1,70776,15.29
2,2,14455,16.22
3,3,4042,16.65
4,4,1956,15.91
5,5,5475,16.59
6,6,3295,18.03


### Aggregation 3 - During which hour are trips longest on average?


In [9]:
duration_by_hour = (
    analysis_df.groupby('Hour')
    .agg(
        avg_trip_duration_min=('trip_duration_min', 'mean'),
        total_trips=('id', 'size')
    )
    .reset_index()
    .sort_values('avg_trip_duration_min', ascending=False)
)

duration_by_hour.round(2)

,Hour,avg_trip_duration_min,total_trips
4,4,18.65,1121
15,15,18.43,4959
16,16,17.74,4340
17,17,17.50,5307
18,18,16.93,6288
14,14,16.91,5161
13,13,16.83,4838
11,11,16.00,4733
12,12,15.85,5095
10,10,15.78,4454



##  Mini EDA

five small business-style questions from the dataset.each output + one-line insight.

### Q1. Which is the peak hour?


In [11]:
trip_by_hour.head(5)

,Hour,total_trips
18,18,6288
19,19,6272
21,21,5849
20,20,5749
22,22,5448


**Insight:** Peak demand is concentrated in the evening, especially 6 PM and 7 PM

### Q2. When do the shortest and longest trips happen?


In [13]:
longest_hours = duration_by_hour[['Hour', 'avg_trip_duration_min', 'total_trips']].head(5)
shortest_hours = duration_by_hour[['Hour', 'avg_trip_duration_min', 'total_trips']].tail(5).sort_values('avg_trip_duration_min')

display(longest_hours.round(2))
display(shortest_hours.round(2))

,Hour,avg_trip_duration_min,total_trips
4,4,18.65,1121
15,15,18.43,4959
16,16,17.74,4340
17,17,17.50,5307
18,18,16.93,6288


,Hour,avg_trip_duration_min,total_trips
6,6,11.86,2245
5,5,12.13,1055
2,2,12.54,1851
7,7,12.91,3792
21,21,13.87,5849


**Insight:** The longest average trips appear mostly around 4 AM and late afternoon, while the shortest averages happen around 5 AM to 7 AM.


### Q3. Do more passengers mean longer trips?


In [14]:
duration_by_passenger.round(2)


,passenger_count,total_trips,mean_trip_duration_min
0,0,1,"1,431.68"
1,1,70776,15.29
2,2,14455,16.22
3,3,4042,16.65
4,4,1956,15.91
5,5,5475,16.59
6,6,3295,18.03


**Insight:** More passengers do not automatically mean much longer trips

### Q4. Do pickup and dropoff locations show any pattern?

In [16]:
pickup_hotspots = analysis_df['pickup_grid'].value_counts().head(5).reset_index()
pickup_hotspots.columns = ['pickup_grid', 'total_trips']

dropoff_hotspots = analysis_df['dropoff_grid'].value_counts().head(5).reset_index()
dropoff_hotspots.columns = ['dropoff_grid', 'total_trips']

display(pickup_hotspots)
display(dropoff_hotspots)

,pickup_grid,total_trips
0,"40.76, -73.97",6223
1,"40.75, -73.99",6148
2,"40.76, -73.98",5320
3,"40.75, -73.98",4850
4,"40.74, -73.99",4453


,dropoff_grid,total_trips
0,"40.76, -73.98",5383
1,"40.76, -73.97",5288
2,"40.75, -73.99",5107
3,"40.75, -73.98",4877
4,"40.74, -73.99",3758


**Insight:** Pickup and dropoff hotspots overlap heavily around a small set of coordinates, which suggests many rides are concentrated in dense central city areas.


### Q5. Do any trips look unusually long or short?


In [18]:
duration_quantiles = analysis_df['trip_duration_min'].quantile([0.01, 0.25, 0.50, 0.75, 0.99]).reset_index()
duration_quantiles.columns = ['quantile', 'trip_duration_min']

shortest_trips = analysis_df.nsmallest(5, 'trip_duration')[['pickup_datetime', 'Hour', 'passenger_count', 'trip_duration_min', 'pickup_grid', 'dropoff_grid']].copy()
longest_trips = analysis_df.nlargest(5, 'trip_duration')[['pickup_datetime', 'Hour', 'passenger_count', 'trip_duration_min', 'pickup_grid', 'dropoff_grid']].copy()

display(duration_quantiles.round(2))
display(shortest_trips.round({'trip_duration_min': 2}))
display(longest_trips.round({'trip_duration_min': 2}))

,quantile,trip_duration_min
0,0.01,1.43
1,0.25,6.60
2,0.50,11.03
3,0.75,17.93
4,0.99,56.47


,pickup_datetime,Hour,passenger_count,trip_duration_min,pickup_grid,dropoff_grid
35196,2016-04-17 11:44:49,11,1,0.02,"40.79, -73.94","40.79, -73.94"
1107,2016-06-23 13:36:48,13,3,0.03,"40.72, -73.83","40.71, -73.82"
6777,2016-05-16 08:20:40,8,1,0.03,"40.77, -73.96","40.77, -73.96"
32760,2016-04-04 06:54:58,6,2,0.03,"40.76, -73.96","40.76, -73.96"
37688,2016-01-12 14:02:29,14,1,0.03,"40.76, -73.98","40.76, -73.98"


,pickup_datetime,Hour,passenger_count,trip_duration_min,pickup_grid,dropoff_grid
73816,2016-05-06 00:00:10,0,1,"1,439.83","40.75, -74.0","40.74, -73.98"
59891,2016-06-30 16:37:52,16,1,"1,439.78","40.75, -73.99","40.8, -73.96"
91717,2016-05-12 13:48:19,13,1,"1,439.63","40.64, -73.78","40.72, -73.98"
66346,2016-05-14 04:48:05,4,1,"1,439.62","40.73, -74.0","40.71, -73.99"
88508,2016-02-22 09:06:37,9,1,"1,439.30","40.78, -73.96","40.81, -73.96"


**Insight:** Yes,trips are almost zero minutes long, while a few are nearly 24 hours, so this dataset likely contains outliers or edge cases.


# Strong observations from the data

- Evening hours, especially around 6 PM and 7 PM, have the highest number of trips.
- some unusual trips that are extremely short or extremely long, so outlier checking would matter before modeling.